# Autoresearch Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.

This notebook is a **public-safe research ledger explorer**. It is useful for seeing how the autonomous loop behaved over time, but it is not the competition-facing source of truth. For the current public metrics, use `docs/CURRENT_FRONTIER.md`, which filters out diagnostic micro-runs and historical metric artifacts.

How to use it: open this notebook in Jupyter or Google Colab from the repository root, make sure `results.tsv` is available beside the notebook, then run the cells top to bottom. The outputs are intentionally cleared in the public repository so judges can regenerate them from the submitted ledger.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings

# Suppress plotting warnings for clean notebook output
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Load the TSV (tab-separated, schema: run_id, state_hash, model, f1_score, latency_ms, vram_gb, total_samples, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["f1_score"] = pd.to_numeric(df["f1_score"], errors="coerce").fillna(0.0)
df["latency_ms"] = pd.to_numeric(df["latency_ms"], errors="coerce").fillna(0.0)
df["vram_gb"] = pd.to_numeric(df["vram_gb"], errors="coerce").fillna(0.0)
df["total_samples"] = pd.to_numeric(df["total_samples"], errors="coerce").fillna(0)
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
# Filter out 0.0 scores as they usually represent pre-flight failures or data issues
kept = df[(df["status"] == "KEEP") & (df["f1_score"] > 0)].copy()
print(f"Valid KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    f1 = row["f1_score"]
    desc = row["description"]
    print(f"  #{i:3d}  f1={f1:.4f}  vram={row['vram_gb']:.2f}GB  latency={row['latency_ms']:.0f}ms  {desc}")

## F1-Score Over Time

Track how the best (kept) f1_score evolves as experiments progress. The running maximum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes and invalid data (F1=0) for plotting
valid = df[(df["status"] != "CRASH") & (df["f1_score"] > 0)].copy()
valid = valid.reset_index(drop=True)

if len(valid) > 0:
    baseline_f1 = valid.loc[0, "f1_score"]

    # Only plot points at or above baseline (the interesting region)
    above = valid[valid["f1_score"] >= baseline_f1 - 0.1]

    # Plot discarded as faint background dots
    disc = above[above["status"] == "DISCARD"]
    ax.scatter(disc.index, disc["f1_score"],
               c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

    # Plot kept experiments as prominent green dots
    kept_v = above[above["status"] == "KEEP"]
    ax.scatter(kept_v.index, kept_v["f1_score"],
               c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

    # Running maximum step line (only improvements)
    # Note: We use the full valid set to find the actual chronological maximums
    running_max = valid["f1_score"].cummax()
    ax.step(valid.index, running_max, where="post", color="#27ae60",
            linewidth=2, alpha=0.7, zorder=3, label="Running best")

    # Label each kept experiment that was an improvement
    current_max = 0
    for i, row in valid[valid["status"] == "KEEP"].iterrows():
        if row["f1_score"] > current_max:
            current_max = row["f1_score"]
            desc = str(row["description"]).strip()
            if len(desc) > 45:
                desc = desc[:42] + "..."

            ax.annotate(desc, (i, row["f1_score"]),
                        textcoords="offset points",
                        xytext=(6, 6), fontsize=8.0,
                        color="#1a7a3a", alpha=0.9,
                        rotation=30, ha="left", va="bottom")

    n_total = len(df)
    n_kept = len(kept_v)
    ax.set_xlabel("Chronological Experiment #", fontsize=12)
    ax.set_ylabel("F1-Score (higher is better)", fontsize=12)
    ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Successful Steps", fontsize=14)
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(True, alpha=0.2)

    # Y-axis: from baseline to just above best
    best_f1 = valid["f1_score"].max()
    margin = max(0.05, (best_f1 - baseline_f1) * 0.15)
    ax.set_ylim(max(0, baseline_f1 - 0.2), best_f1 + margin)

    plt.tight_layout()
    plt.savefig("progress.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to progress.png")
else:
    print("No valid data to plot.")

## Summary Statistics

In [ ]:
# Summary stats (excluding invalid data)
valid_kept = df[(df["status"] == "KEEP") & (df["f1_score"] > 0)].copy()
if len(valid_kept) > 0:
    baseline_f1 = df.iloc[0]["f1_score"]
    best_f1 = valid_kept["f1_score"].max()
    best_row = valid_kept.loc[valid_kept["f1_score"].idxmax()]

    print(f"Baseline F1-Score:  {baseline_f1:.4f}")
    print(f"Best F1-Score:      {best_f1:.4f}")
    print(f"Total improvement: {best_f1 - baseline_f1:+.4f} ({(best_f1 - baseline_f1) / max(0.0001, baseline_f1) * 100:.2f}%)")
    print(f"Best model:         {best_row['model']}")
    print(f"Best experiment:    {best_row['description']}")
    print()

    # How many experiments to find each improvement
    print("Cumulative effort per successful improvement:")
    current_best = 0
    for i, row in df[df["status"] == "KEEP"].iterrows():
        if row["f1_score"] > current_best:
            current_best = row["f1_score"]
            desc = str(row["description"]).strip()
            print(f"  Experiment #{i:3d}: f1={row['f1_score']:.4f}  {desc}")
else:
    print("No valid experiments found.")

## Top Hits (Significant Improvements)

This section shows raw chronological improvements in the ledger. Some top raw rows are micro-runs or diagnostic artifacts; validate public claims against the full-50 frontier in `docs/CURRENT_FRONTIER.md`.


In [ ]:
# Each experiment's contribution is its improvement over the *previous best*
significant = []
current_best = df.iloc[0]["f1_score"]

for i, row in df.iterrows():
    if row["f1_score"] > current_best:
        delta = row["f1_score"] - current_best
        current_best = row["f1_score"]
        significant.append({
            "index": i,
            "delta": delta,
            "f1_score": row["f1_score"],
            "model": row["model"],
            "description": row["description"]
        })

hits = pd.DataFrame(significant)
if len(hits) > 0:
    hits = hits.sort_values("delta", ascending=False)

    print(f"{'Rank':>4}  {'Delta':>8}  {'F1':>10}  {'Model':<30}  Description")
    print("-" * 100)
    for rank, (_, row) in enumerate(hits.iterrows(), 1):
        print(f"{rank:4d}  {row['delta']:+.4f}  {row['f1_score']:.4f}  {str(row['model']):<30}  {row['description']}")

    print(f"\n{'':>4}  {hits['delta'].sum():+.4f}  {'':>10}  PEAK improvement over baseline")
else:
    print("No significant improvements recorded over baseline.")